# Fine-Tuned Basic Pitch — Leave-One-Player-Out Cross-Validation

Rigorous version: trains **6 models**, each holding out one player as test, to put error bars on the fine-tuning gain.

**Per fold k:** test = player k, validation = player (k+1)%6, train = the other 4 players.
This rotates the exact 4-train / 1-val / 1-test structure from the single-split run through all 6 test positions. Fully leakage-free: thresholds tuned on the val player, reported on the test player, neither in that fold's training set.

**Both pretrained and fine-tuned** evaluated identically per fold (same overlapping-window inference, same per-fold threshold tuning). Pretrained outputs are cached once (they don't change across folds).

**Runtime: ~30-35 min.** TFRecord conversion (~10 min, one-time) → cache pretrained (~5 min) → 6 folds (~3 min each).


## 1. Environment + Basic Pitch patches

In [1]:
!nvidia-smi -L
import os
if not os.path.exists('/content/basic-pitch'):
    !git clone -q https://github.com/spotify/basic-pitch.git /content/basic-pitch
!pip install -q --no-deps -e /content/basic-pitch 2>/dev/null
!pip install -q librosa soundfile sox mirdata tensorflow mir_eval resampy==0.4.2
!apt-get install -y sox > /dev/null 2>&1

import sys; sys.path.insert(0,'/content/basic-pitch')
import warnings; warnings.filterwarnings('ignore')
import tensorflow as tf
print(f"TF: {tf.__version__}")

# TF2.20/Keras3 patches
p='/content/basic-pitch/basic_pitch/layers/signal.py'
open(p,'w').write(open(p).read().replace('rank = input_shape.rank','rank = len(input_shape)'))
p='/content/basic-pitch/basic_pitch/models.py'; s=open(p).read()
s=s.replace('x = tf.expand_dims(x, -1)\n    if use_batchnorm:','x = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x)\n    if use_batchnorm:')
s=s.replace('x_contours_reduced = tf.expand_dims(x_contours, -1)','x_contours_reduced = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x_contours)')
open(p,'w').write(s)
p='/content/basic-pitch/basic_pitch/nn.py'
open(p,'w').write(open(p).read().replace('tf.debugging.assert_equal(tf.shape(x).shape, 4)','pass'))

import importlib, basic_pitch.layers.signal as _s, basic_pitch.nn as _n, basic_pitch.models as _m
importlib.reload(_s); importlib.reload(_n); importlib.reload(_m)
import basic_pitch.models as bp_models
print("Patches applied.")

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-0833beed-306b-b682-5f6d-69cea722b4cf)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basic-pitch (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 110.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.8/263.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━

TF: 2.20.0
Patches applied.


## 2. Drive + data + TFRecords (per-player layout)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import glob, shutil, json
import numpy as np
from pathlib import Path

DATA_ROOT = next((c for c in [Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
                              Path('/content/drive/MyDrive/FullGuitarSetData')]
                  if (c/'JamsFiles').exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("GuitarSet not found")
LOCAL_AUDIO,LOCAL_JAMS='/content/gs_audio','/content/gs_jams'
os.makedirs(LOCAL_AUDIO,exist_ok=True); os.makedirs(LOCAL_JAMS,exist_ok=True)
def copy_if_needed(src,dst,ext):
    s=glob.glob(os.path.join(str(src),f'*.{ext}'))
    if len(glob.glob(os.path.join(dst,f'*.{ext}')))>=len(s): print(f"  {ext}: on SSD"); return
    print(f"  copying {len(s)} {ext}...")
    for f in s:
        try: shutil.copy2(f,dst)
        except shutil.SameFileError: pass
copy_if_needed(DATA_ROOT/'AudioFiles',LOCAL_AUDIO,'wav')
copy_if_needed(DATA_ROOT/'JamsFiles',LOCAL_JAMS,'jams')

# --- TFRecord conversion to per-player dirs (one-time) ---
import sox
from basic_pitch.constants import (AUDIO_SAMPLE_RATE, AUDIO_N_CHANNELS, ANNOTATION_HOP,
    FREQ_BINS_NOTES, FREQ_BINS_CONTOURS, N_FREQ_BINS_NOTES, N_FREQ_BINS_CONTOURS)
from basic_pitch.data.tf_example_serialization import bytes_feature

TFR_ROOT='/content/gs_tfr_byplayer'; os.makedirs(TFR_ROOT,exist_ok=True)

def jams_notes(jp):
    jam=json.load(open(jp)); out=[]
    for ann in jam.get('annotations',[]):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for o in ann['data']:
            midi=float(o['value']); freq=440.0*(2.0**((midi-69.0)/12.0))
            out.append({'onset':float(o['time']),'offset':float(o['time'])+float(o['duration']),'freq':freq})
    return sorted(out,key=lambda n:n['onset'])

def to_sparse(notes,ts,fbins,onsets_only=False):
    idx,val=[],[]
    edges=np.concatenate([[0],(fbins[:-1]+fbins[1:])/2,[fbins[-1]*2]])
    for n in notes:
        b=int(np.clip(np.searchsorted(edges,n['freq'])-1,0,len(fbins)-1))
        if onsets_only:
            t=int(round(n['onset']/ANNOTATION_HOP))
            if 0<=t<len(ts): idx.append([t,b]); val.append(1.0)
        else:
            t0=int(round(n['onset']/ANNOTATION_HOP)); t1=int(round(n['offset']/ANNOTATION_HOP))
            for t in range(max(0,t0),min(len(ts),t1)): idx.append([t,b]); val.append(1.0)
    if not idx: return np.zeros((0,2),np.int64),np.zeros(0,np.float32)
    return np.array(idx,np.int64),np.array(val,np.float32)

def make_tfr(jp,ap,player):
    stem=os.path.splitext(os.path.basename(jp))[0]
    outp=os.path.join(TFR_ROOT,player,f'{stem}.tfrecord')
    if os.path.exists(outp): return
    tmp=f'/tmp/{stem}_22k.wav'
    if not os.path.exists(tmp):
        tfm=sox.Transformer(); tfm.rate(AUDIO_SAMPLE_RATE); tfm.channels(AUDIO_N_CHANNELS); tfm.build(ap,tmp)
    dur=sox.file_info.duration(tmp)
    ts=np.arange(0,dur+ANNOTATION_HOP,ANNOTATION_HOP); ntf=len(ts)
    notes=jams_notes(jp)
    ni,nv=to_sparse(notes,ts,FREQ_BINS_NOTES); oi,ov=to_sparse(notes,ts,FREQ_BINS_NOTES,True)
    ci,cv=to_sparse(notes,ts,FREQ_BINS_CONTOURS)
    ex=tf.train.Example(features=tf.train.Features(feature={
        'file_id':bytes_feature(bytes(stem,'utf-8')),'source':bytes_feature(b'guitarset'),
        'audio_wav':bytes_feature(open(tmp,'rb').read()),
        'notes_indices':bytes_feature(tf.io.serialize_tensor(ni).numpy()),
        'notes_values':bytes_feature(tf.io.serialize_tensor(nv).numpy()),
        'onsets_indices':bytes_feature(tf.io.serialize_tensor(oi).numpy()),
        'onsets_values':bytes_feature(tf.io.serialize_tensor(ov).numpy()),
        'contours_indices':bytes_feature(tf.io.serialize_tensor(ci).numpy()),
        'contours_values':bytes_feature(tf.io.serialize_tensor(cv).numpy()),
        'notes_onsets_shape':bytes_feature(tf.io.serialize_tensor(np.array([ntf,N_FREQ_BINS_NOTES],np.int64)).numpy()),
        'contours_shape':bytes_feature(tf.io.serialize_tensor(np.array([ntf,N_FREQ_BINS_CONTOURS],np.int64)).numpy()),
    }))
    os.makedirs(os.path.dirname(outp),exist_ok=True)
    with tf.io.TFRecordWriter(outp) as w: w.write(ex.SerializeToString())

PLAYERS=['00','01','02','03','04','05']
existing=sum(len(glob.glob(os.path.join(TFR_ROOT,p,'*.tfrecord'))) for p in PLAYERS)
if existing>=360:
    print(f"TFRecords already present ({existing}).")
else:
    jamses=sorted(glob.glob(LOCAL_JAMS+'/*.jams'))
    print(f"Converting {len(jamses)} recordings...")
    for i,jp in enumerate(jamses):
        stem=os.path.splitext(os.path.basename(jp))[0]; player=stem.split('_')[0]
        cands=(glob.glob(os.path.join(LOCAL_AUDIO,stem+'*mic*.wav')) or glob.glob(os.path.join(LOCAL_AUDIO,stem+'*.wav')))
        if not cands: continue
        try: make_tfr(jp,cands[0],player)
        except Exception as e:
            if i<3: print(f"  err {stem}: {e}")
        if (i+1)%90==0: print(f"  {i+1}/{len(jamses)}")
for p in PLAYERS: print(f"  player {p}: {len(glob.glob(os.path.join(TFR_ROOT,p,'*.tfrecord')))} tfrecords")

Mounted at /content/drive
  copying 360 wav...
  copying 360 jams...
Converting 360 recordings...
  90/360
  180/360
  270/360
  360/360
  player 00: 60 tfrecords
  player 01: 60 tfrecords
  player 02: 60 tfrecords
  player 03: 60 tfrecords
  player 04: 60 tfrecords
  player 05: 60 tfrecords


## 3. Inference + eval harness (shared by both models)

In [3]:
import pandas as pd, librosa
from basic_pitch.constants import AUDIO_SAMPLE_RATE, AUDIO_N_SAMPLES, FFT_HOP, ANNOTATIONS_FPS
from basic_pitch.note_creation import model_output_to_notes

N_OL=30; OVERLAP_LEN=N_OL*FFT_HOP; HOP_SIZE=AUDIO_N_SAMPLES-OVERLAP_LEN
MIN_NOTE_LEN=int(np.round(58/1000*ANNOTATIONS_FPS))

def window_audio(a):
    a=np.concatenate([np.zeros(OVERLAP_LEN//2,np.float32),a.astype(np.float32)]); out=[]; st=0
    while st<len(a):
        w=a[st:st+AUDIO_N_SAMPLES]
        if len(w)<AUDIO_N_SAMPLES: w=np.pad(w,(0,AUDIO_N_SAMPLES-len(w)))
        out.append(w)
        if st+AUDIO_N_SAMPLES>=len(a): break
        st+=HOP_SIZE
    return np.stack(out)

def unwrap(st,orig):
    n=N_OL//2; tr=st[:,n:-n,:]; flat=tr.reshape(-1,tr.shape[-1])
    return flat[:int(np.floor(orig*(ANNOTATIONS_FPS/AUDIO_SAMPLE_RATE))),:]

def infer(model,ap,is_saved):
    y,_=librosa.load(ap,sr=AUDIO_SAMPLE_RATE,mono=True)
    x=tf.constant(window_audio(y)[...,None],tf.float32)
    out=(model.signatures['serving_default'](input_2=x) if is_saved else model(x,training=False))
    return {'onset':unwrap(out['onset'].numpy(),len(y)),'note':unwrap(out['note'].numpy(),len(y)),
            'contour':unwrap(out['contour'].numpy(),len(y))}

def load_gt(jp):
    jam=json.load(open(jp)); notes=[]
    for ann in jam.get('annotations',[]):
        if ann.get('namespace','') not in ('note_midi','pitch_midi'): continue
        for o in ann['data']:
            notes.append({'onset':float(o['time']),'offset':float(o['time'])+float(o['duration']),'midi':int(round(float(o['value'])))})
    return sorted(notes,key=lambda n:n['onset'])

def match(gt,pred,tol=0.05):
    c=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p['midi'])==int(g['midi']) and abs(p['onset']-g['onset'])<=tol: c.append((abs(p['onset']-g['onset']),pi,gi))
    c.sort(); up,ug=set(),set()
    for _,pi,gi in c:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0; R=tp/(tp+fn) if tp+fn else 0
    return P,R,(2*P*R/(P+R) if P+R else 0)

def cache_player(model,is_saved,player):
    jams=[j for j in sorted(glob.glob(LOCAL_JAMS+'/*.jams')) if os.path.basename(j).split('_')[0]==player]
    out={}
    for jp in jams:
        stem=os.path.splitext(os.path.basename(jp))[0]
        cands=(glob.glob(os.path.join(LOCAL_AUDIO,stem+'*mic*.wav')) or glob.glob(os.path.join(LOCAL_AUDIO,stem+'*.wav')))
        if not cands: continue
        out[stem]={**infer(model,cands[0],is_saved),'gt':load_gt(jp)}
    return out

def eval_cache(cache,ot,ft):
    rows=[]
    for d in cache.values():
        _,ev=model_output_to_notes({'onset':d['onset'],'note':d['note'],'contour':d['contour']},
            onset_thresh=ot,frame_thresh=ft,min_note_len=MIN_NOTE_LEN,min_freq=None,max_freq=None,include_pitch_bends=False)
        pred=[{'onset':float(n[0]),'offset':float(n[1]),'midi':int(n[2])} for n in ev]
        P,R,F=match(d['gt'],pred); rows.append({'P50':P,'R50':R,'F50':F,'n':len(d['gt'])})
    df=pd.DataFrame(rows)
    return {k:float(np.average(df[k],weights=df.n)) for k in ['P50','R50','F50']}

ONSETS=[0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]; FRAMES=[0.10,0.20,0.30]
def best_threshold(cache):
    best=None
    for ot in ONSETS:
        for ft in FRAMES:
            a=eval_cache(cache,ot,ft)
            if best is None or a['F50']>best['F50']: best={**a,'onset_t':ot,'frame_t':ft}
    return best
print("Harness ready.")

Harness ready.


## 4. Cache pretrained baseline once (all 6 players)

In [4]:
from basic_pitch import ICASSP_2022_MODEL_PATH
pretrained=tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
PRE={}
for p in PLAYERS:
    print(f"Caching pretrained player {p}...")
    PRE[p]=cache_player(pretrained,True,p)
print("Pretrained cached for all players.")

Caching pretrained player 00...
Caching pretrained player 01...
Caching pretrained player 02...
Caching pretrained player 03...
Caching pretrained player 04...
Caching pretrained player 05...
Pretrained cached for all players.


## 5. Leave-one-player-out loop
Trains a fresh fine-tuned model per fold. Same weight-transfer + weighted-loss recipe (`pos_weight=5`, label smoothing 0.05) that gave the working single-split result.

In [8]:
import importlib
import basic_pitch.layers.signal
import basic_pitch.nn
import basic_pitch.models
importlib.reload(basic_pitch.layers.signal)
importlib.reload(basic_pitch.nn)
importlib.reload(basic_pitch.models)
import basic_pitch.models as bp_models

# Verify the module is whole again
print("FlattenAudioCh present:", hasattr(basic_pitch.nn, 'FlattenAudioCh'))
print("model builds:", bp_models.model() is not None)

FlattenAudioCh present: False


AttributeError: module 'basic_pitch.nn' has no attribute 'FlattenAudioCh'

In [10]:
!cd /content/basic-pitch && git checkout basic_pitch/nn.py
print("Restored. Size:", len(open('/content/basic-pitch/basic_pitch/nn.py').read()))

Updated 1 path from the index
Restored. Size: 4098


In [11]:
p = '/content/basic-pitch/basic_pitch/nn.py'
src = open(p).read()
assert 'FlattenAudioCh' in src, "file still broken"
if 'tf.debugging.assert_equal(tf.shape(x).shape, 4)' in src:
    src = src.replace('tf.debugging.assert_equal(tf.shape(x).shape, 4)', 'pass')
    open(p, 'w').write(src)
    print("Patched nn.py")
else:
    print("Already patched or line not found — checking FlattenAudioCh still present:", 'FlattenAudioCh' in open(p).read())

Patched nn.py


In [13]:
# Restore signal.py
!cd /content/basic-pitch && git checkout basic_pitch/layers/signal.py

p = '/content/basic-pitch/basic_pitch/layers/signal.py'
src = open(p).read()
assert 'NormalizedLog' in src, "still broken"
if 'rank = input_shape.rank' in src:
    src = src.replace('rank = input_shape.rank', 'rank = len(input_shape)')
    open(p, 'w').write(src)
    print("Patched signal.py")
else:
    print("Already patched, NormalizedLog present:", 'NormalizedLog' in open(p).read())

Updated 1 path from the index
Patched signal.py


In [14]:
p = '/content/basic-pitch/basic_pitch/models.py'
src = open(p).read()
print(f"models.py: {len(src)} bytes")
if 'def model(' not in src or len(src) < 1000:
    print("WIPED — restoring and repatching")
    import subprocess
    subprocess.run(['git','checkout','basic_pitch/models.py'], cwd='/content/basic-pitch')
    src = open(p).read()
    src = src.replace('x = tf.expand_dims(x, -1)\n    if use_batchnorm:',
                      'x = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x)\n    if use_batchnorm:')
    src = src.replace('x_contours_reduced = tf.expand_dims(x_contours, -1)',
                      'x_contours_reduced = tfkl.Lambda(lambda t: tf.expand_dims(t, -1))(x_contours)')
    open(p,'w').write(src)
    print("Restored + repatched models.py")
else:
    print("models.py OK")

models.py: 10675 bytes
models.py OK


In [15]:
import importlib
import basic_pitch.layers.signal, basic_pitch.nn, basic_pitch.models
importlib.reload(basic_pitch.layers.signal)
importlib.reload(basic_pitch.nn)
importlib.reload(basic_pitch.models)
import basic_pitch.models as bp_models
print("model builds:", bp_models.model() is not None)

model builds: True


In [16]:
# Weight transfer (verified shape-sort method)
def transfer_pretrained(saved, kmodel):
    order={'kernel':0,'bias':1,'gamma':2,'beta':3,'moving_mean':4,'moving_variance':5}
    def sk(v):
        name=v.name if hasattr(v,'name') else v.path
        param=name.split('/')[-1].replace(':0','').split('_')[0]
        return (str(list(v.shape)), order.get(param,9))
    for s,k in zip(sorted(saved.variables,key=sk), sorted(kmodel.variables,key=sk)):
        k.assign(s)

def weighted_bce(y_true,y_pred):
    bce=tf.keras.losses.binary_crossentropy(y_true,y_pred,label_smoothing=0.05)
    w=1.0+4.0*tf.reduce_mean(y_true,axis=-1)
    return tf.reduce_mean(w*bce)

from basic_pitch.data.tf_example_deserialization import prepare_datasets
BP_DATA='/content/bp_lopo'

def rebuild_splits(test_p, val_p):
    root=os.path.join(BP_DATA,'guitarset','splits')
    if os.path.exists(root): shutil.rmtree(root)
    for sp in ['train','validation','test']: os.makedirs(os.path.join(root,sp),exist_ok=True)
    train_ps=[p for p in PLAYERS if p not in (test_p,val_p)]
    for p in train_ps:
        for f in glob.glob(os.path.join(TFR_ROOT,p,'*.tfrecord')):
            os.symlink(f, os.path.join(root,'train',os.path.basename(f)))
    for f in glob.glob(os.path.join(TFR_ROOT,val_p,'*.tfrecord')):
        os.symlink(f, os.path.join(root,'validation',os.path.basename(f)))
    for f in glob.glob(os.path.join(TFR_ROOT,test_p,'*.tfrecord')):
        os.symlink(f, os.path.join(root,'test',os.path.basename(f)))

fold_results=[]
for k,test_p in enumerate(PLAYERS):
    val_p=PLAYERS[(k+1)%6]
    train_ps=[p for p in PLAYERS if p not in (test_p,val_p)]
    print(f"\n===== FOLD {k}: test={test_p}  val={val_p}  train={train_ps} =====")

    rebuild_splits(test_p,val_p)

    train_ds,val_ds=prepare_datasets(BP_DATA,training_shuffle_buffer_size=100,batch_size=8,
        validation_steps=20,datasets_to_use=['guitarset'],dataset_sampling_frequency=np.array([1.0]))
    train_ds=train_ds.map(lambda x,y,w:(x,y)); val_ds=val_ds.map(lambda x,y,w:(x,y))
    model=bp_models.model()
    transfer_pretrained(pretrained,model)
    model.compile(loss={'onset':weighted_bce,'note':weighted_bce,'contour':weighted_bce},
                  optimizer=tf.keras.optimizers.Adam(1e-4))
    model.fit(train_ds,epochs=50,steps_per_epoch=100,validation_data=val_ds,validation_steps=20,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=10,restore_best_weights=True),
                         tf.keras.callbacks.ReduceLROnPlateau(patience=5,factor=0.5)],verbose=0)
    print("  trained. caching val+test...")

    ft_val=cache_player(model,False,val_p); ft_test=cache_player(model,False,test_p)
    bft=best_threshold(ft_val);  rft=eval_cache(ft_test,bft['onset_t'],bft['frame_t'])
    bpre=best_threshold(PRE[val_p]); rpre=eval_cache(PRE[test_p],bpre['onset_t'],bpre['frame_t'])

    fold_results.append({'fold':k,'test':test_p,
        'pre_F50':rpre['F50'],'pre_P50':rpre['P50'],'pre_R50':rpre['R50'],'pre_thr':(bpre['onset_t'],bpre['frame_t']),
        'ft_F50':rft['F50'],'ft_P50':rft['P50'],'ft_R50':rft['R50'],'ft_thr':(bft['onset_t'],bft['frame_t']),
        'delta_F50':rft['F50']-rpre['F50']})
    print(f"  pretrained F50={rpre['F50']:.4f} (thr {bpre['onset_t']}/{bpre['frame_t']}) | "
          f"fine-tuned F50={rft['F50']:.4f} (thr {bft['onset_t']}/{bft['frame_t']}) | delta={rft['F50']-rpre['F50']:+.4f}")
    del ft_val, ft_test, model
print("\nAll folds done.")


===== FOLD 0: test=00  val=01  train=['02', '03', '04', '05'] =====


Cause: could not parse the source code of <function <lambda> at 0x7eceff941120>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7eceff941120>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7eceff940680>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7eceff940680>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
  trained. caching val+test...
  pretrained F50=0.7281 (thr 0.5/0.3) | fine-tuned F50=0.7180 (thr 0.1/0.3) | delta=-0.0101

===== FOLD 1: test=01  val=02  train=['00', '03', '04', '05'] =====


...

Cause: could not parse the source code of <function <lambda> at 0x7ecf00035760>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
  trained. caching val+test...
  pretrained F50=0.7168 (thr 0.5/0.3) | fine-tuned F50=0.7302 (thr 0.1/0.3) | delta=+0.0134

===== FOLD 5: test=05  val=00  train=['01', '02', '03', '04'] =====


Cause: could not parse the source code of <function <lambda> at 0x7ecf217a6480>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7ecf217a6480>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7eceff883f60>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: could not parse the source code of <function <lambda> at 0x7eceff883f60>: found multiple definitions with identical signatures at the location. This error may be avoided by defining each lambda on a single line and with unique argument names. The matching definitions were:
Match 0:
lambda x, y, w: (x, y)

Match 1:
lambda x, y, w: (x, y)

To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
  trained. caching val+test...
  pretrained F50=0.7705 (thr 0.5/0.3) | fine-tuned F50=0.7918 (thr 0.1/0.3) | delta=+0.0213

All folds done.


## 6. Aggregate — mean ± std across folds

In [17]:
df=pd.DataFrame(fold_results)
print("Per-fold results:")
print(df[['fold','test','pre_F50','ft_F50','delta_F50']].to_string(index=False))

def ms(col): return df[col].mean(), df[col].std()
pf=ms('pre_F50'); ff=ms('ft_F50'); d=ms('delta_F50')
pp=ms('pre_P50'); fp=ms('ft_P50'); pr=ms('pre_R50'); fr=ms('ft_R50')

print("\n"+"="*60)
print("LEAVE-ONE-PLAYER-OUT CV  (mean +/- std over 6 folds)")
print("="*60)
print(f"{'':24s} {'P50':>15} {'R50':>15} {'F50':>15}")
print(f"{'Pretrained Basic Pitch':24s} {pp[0]:.3f}+/-{pp[1]:.3f}  {pr[0]:.3f}+/-{pr[1]:.3f}  {pf[0]:.3f}+/-{pf[1]:.3f}")
print(f"{'Fine-tuned Basic Pitch':24s} {fp[0]:.3f}+/-{fp[1]:.3f}  {fr[0]:.3f}+/-{fr[1]:.3f}  {ff[0]:.3f}+/-{ff[1]:.3f}")
print("-"*60)
print(f"{'Delta F50':24s} {d[0]:+.4f} +/- {d[1]:.4f}  (across {len(df)} folds)")
print(f"{'Folds with positive delta':24s} {(df.delta_F50>0).sum()}/{len(df)}")
print("="*60)

Per-fold results:
 fold test  pre_F50   ft_F50  delta_F50
    0   00 0.728065 0.717976  -0.010089
    1   01 0.772972 0.792799   0.019827
    2   02 0.723177 0.696777  -0.026400
    3   03 0.741730 0.756233   0.014503
    4   04 0.716849 0.730200   0.013351
    5   05 0.770452 0.791800   0.021348

LEAVE-ONE-PLAYER-OUT CV  (mean +/- std over 6 folds)
                                     P50             R50             F50
Pretrained Basic Pitch   0.682+/-0.020  0.838+/-0.045  0.742+/-0.024
Fine-tuned Basic Pitch   0.732+/-0.018  0.783+/-0.065  0.748+/-0.040
------------------------------------------------------------
Delta F50                +0.0054 +/- 0.0193  (across 6 folds)
Folds with positive delta 4/6
